In [15]:
import os
import warnings
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings("ignore")

DATA_DIR = os.environ.get("DATA_DIR", ".")
OUT_DIR = os.environ.get("OUT_DIR", "outputs")
os.makedirs(OUT_DIR, exist_ok=True)

# ---- Time split ----
TRAIN_END = "2012-01-01"   # train: before this date, test: from this date
SIM_START = "2012-06-01"   # test period is split: calibration (before) / simulation (from)

# ---- Model selection ----
# None = pick the best model automatically by calibration-period WAPE (recommended).
# Set to a model name (e.g. "RandomForest") to force it; the output will then say it was chosen manually.
MODEL_OVERRIDE = None

# ---- Supply chain ASSUMPTIONS (not in the dataset; change and state them in your resume/interview) ----
LEAD_TIME_WEEKS = 2        # supplier lead time (must be >= 1)
Z = 1.65                   # ~95% cycle service level
UNIT_PRICE = 25.0          # assumed average selling price per unit ($) -> converts sales $ to units
HOLDING_RATE = 0.25        # annual holding cost as a fraction of unit value
ORDER_COST = 100.0         # fixed cost per order ($)
STOCKOUT_COST = 0.4 * UNIT_PRICE   # penalty per unit of lost sales ($)
H = HOLDING_RATE * UNIT_PRICE      # holding cost per unit per year ($)

## 1. Load and prepare data

In [17]:
train = pd.read_csv(os.path.join(DATA_DIR, "train.csv"), parse_dates=["Date"])
features = pd.read_csv(os.path.join(DATA_DIR, "features.csv"), parse_dates=["Date"])
stores = pd.read_csv(os.path.join(DATA_DIR, "stores.csv"))

data = train.merge(features, on=["Store", "Date", "IsHoliday"], how="left")
data = data.merge(stores, on="Store", how="left")
data = data.sort_values(["Store", "Dept", "Date"]).reset_index(drop=True)

markdown_cols = ["MarkDown1", "MarkDown2", "MarkDown3", "MarkDown4", "MarkDown5"]
data[markdown_cols] = data[markdown_cols].fillna(0)

for c in ["CPI", "Unemployment"]:
    data[c] = data.groupby("Store")[c].transform(lambda s: s.ffill().bfill())

data["IsHoliday"] = data["IsHoliday"].astype(int)
data["Type"] = data["Type"].map({"A": 0, "B": 1, "C": 2})
data["Year"] = data["Date"].dt.year
data["Month"] = data["Date"].dt.month
data["Week"] = data["Date"].dt.isocalendar().week.astype(int)

# Returns create negative weekly sales; treat as zero demand
n_neg = int((data["Weekly_Sales"] < 0).sum())
data["Sales_Pos"] = data["Weekly_Sales"].clip(lower=0)

print(f"Rows: {len(data):,} | negative-sales rows clipped to 0: {n_neg}")


# =====================================================
# HISTORICAL DEMAND FEATURES
# =====================================================

group = data.groupby(["Store", "Dept"])["Sales_Pos"]

# Previous sales
data["Lag_1"] = group.shift(1)
data["Lag_4"] = group.shift(4)
data["Lag_13"] = group.shift(13)
data["Lag_52"] = group.shift(52)

# Rolling averages of previous demand
data["Rolling_4"] = group.transform(
    lambda x: x.shift(1).rolling(4).mean()
)

data["Rolling_12"] = group.transform(
    lambda x: x.shift(1).rolling(12).mean()
)


# =====================================================
# FEATURES USED BY ML MODELS
# =====================================================

features_used = [
    "Store", "Dept", "IsHoliday", "Type", "Size",
    "Temperature", "Fuel_Price", "CPI", "Unemployment",
    "Year", "Month", "Week"
] + markdown_cols + [
    "Lag_1", "Lag_4", "Lag_13", "Lag_52",
    "Rolling_4", "Rolling_12"
]


# Remove rows where historical features don't exist
data = data.dropna(subset=[
    "Lag_1", "Lag_4", "Lag_13", "Lag_52",
    "Rolling_4", "Rolling_12"
])


# =====================================================
# TRAIN / TEST SPLIT
# =====================================================

train_data = data[data["Date"] < TRAIN_END].copy()
test_data = data[data["Date"] >= TRAIN_END].copy()

print(f"Train: {len(train_data):,} rows | Test: {len(test_data):,} rows")

Rows: 421,570 | negative-sales rows clipped to 0: 1285
Train: 136,342 rows | Test: 124,741 rows


## 2. Forecasting: compare every model, then select the best

In [18]:
def metrics(y, p):
    y, p = np.asarray(y, float), np.asarray(p, float)
    return {
        "MAE": mean_absolute_error(y, p),
        "RMSE": float(np.sqrt(mean_squared_error(y, p))),
        "WAPE_%": float(np.abs(y - p).sum() / np.abs(y).sum() * 100),
        "R2": r2_score(y, p),
    }

# Naive baseline: sales of the same store-dept in the same week last year (364 days earlier)
ly = data[["Store", "Dept", "Date", "Sales_Pos"]].copy()
ly["Date"] = ly["Date"] + pd.Timedelta(days=364)
ly = ly.rename(columns={"Sales_Pos": "Naive_LY"})
test_data = test_data.merge(ly, on=["Store", "Dept", "Date"], how="left")

X_train, y_train = train_data[features_used], train_data["Sales_Pos"]
X_test = test_data[features_used]

models = {
    "LinearRegression": make_pipeline(StandardScaler(), LinearRegression()),
    "DecisionTree": DecisionTreeRegressor(min_samples_leaf=5, random_state=42),
    "KNN": make_pipeline(StandardScaler(), KNeighborsRegressor(n_neighbors=5, n_jobs=-1)),
    "RandomForest": RandomForestRegressor(n_estimators=100, min_samples_leaf=2, random_state=42, n_jobs=-1),
    "HistGradientBoosting": HistGradientBoostingRegressor(max_iter=300, learning_rate=0.08, random_state=42),
}
try:  # optional: only used if xgboost is installed (pip install xgboost)
    from xgboost import XGBRegressor
    models["XGBoost"] = XGBRegressor(n_estimators=300, learning_rate=0.08, max_depth=8,
                                     subsample=0.8, random_state=42, n_jobs=-1)
except ImportError:
    print("xgboost not installed; skipping it")

for name, m in models.items():
    print(f"Training {name} ...")
    m.fit(X_train, y_train)
    test_data["Pred_" + name] = np.clip(m.predict(X_test), 0, None)

# ---- Compare EVERY model on the 2012 holdout (same rows for all, naive baseline included) ----
cmp_mask = test_data["Naive_LY"].notna()
rows = {"Naive (last year)": metrics(test_data.loc[cmp_mask, "Sales_Pos"], test_data.loc[cmp_mask, "Naive_LY"])}
for name in models:
    rows[name] = metrics(test_data.loc[cmp_mask, "Sales_Pos"], test_data.loc[cmp_mask, "Pred_" + name])
comparison = pd.DataFrame(rows).T.sort_values("WAPE_%").round(3)
print("\nModel comparison on 2012 holdout (sorted by WAPE, lower is better):")
print(comparison)

# ---- Select the final model on the CALIBRATION part of the test period (before SIM_START) ----
calib_mask = test_data["Date"] < SIM_START
calib_wape = {n: metrics(test_data.loc[calib_mask, "Sales_Pos"], test_data.loc[calib_mask, "Pred_" + n])["WAPE_%"]
              for n in models}
print("\nCalibration-period WAPE %:", {k: round(v, 2) for k, v in sorted(calib_wape.items(), key=lambda kv: kv[1])})
auto_best = min(calib_wape, key=calib_wape.get)
if MODEL_OVERRIDE is None:
    best_name = auto_best
    print(f"\n>>> BEST MODEL: {best_name}")
else:
    best_name = MODEL_OVERRIDE
    print(f"\n>>> MODEL CHOSEN MANUALLY: {best_name} (best by metric was {auto_best}). "
          f"Report this honestly: the choice was not made on accuracy alone.")
test_data["Pred_Sales"] = test_data["Pred_" + best_name]

# Feature importance (Random Forest)
imp = pd.Series(models["RandomForest"].feature_importances_, index=features_used).sort_values(ascending=False)
print("\nTop 10 features (Random Forest):")
print(imp.head(10).round(4))

Training LinearRegression ...
Training DecisionTree ...
Training KNN ...
Training RandomForest ...
Training HistGradientBoosting ...
Training XGBoost ...

Model comparison on 2012 holdout (sorted by WAPE, lower is better):
                           MAE      RMSE  WAPE_%     R2
RandomForest          1369.062  2957.355   8.191  0.983
XGBoost               1405.665  3058.944   8.410  0.981
HistGradientBoosting  1459.892  3209.994   8.735  0.980
LinearRegression      1742.067  3444.101  10.423  0.977
DecisionTree          1819.867  3700.449  10.889  0.973
Naive (last year)     1873.983  3944.238  11.212  0.969
KNN                   2195.880  4195.071  13.138  0.965

Calibration-period WAPE %: {'RandomForest': 9.67, 'XGBoost': 10.08, 'HistGradientBoosting': 10.32, 'LinearRegression': 12.33, 'DecisionTree': 12.49, 'KNN': 15.49}

>>> BEST MODEL: RandomForest

Top 10 features (Random Forest):
Lag_52        0.8111
Lag_1         0.1278
Rolling_4     0.0379
Rolling_12    0.0066
Lag_4         0.0

## 3. Convert to units and derive inventory parameters
Safety stock = Z x RMSE of forecast error (calibration period) x sqrt(lead time).

In [19]:
test_data["Actual_Units"] = test_data["Sales_Pos"] / UNIT_PRICE
test_data["Pred_Units"] = test_data["Pred_Sales"] / UNIT_PRICE
test_data["Error_Units"] = test_data["Actual_Units"] - test_data["Pred_Units"]

calib = test_data[test_data["Date"] < SIM_START]
sim = test_data[test_data["Date"] >= SIM_START]

hist_mean = (train_data[train_data["Date"] >= "2011-01-01"]
             .assign(U=lambda d: d["Sales_Pos"] / UNIT_PRICE)
             .groupby(["Store", "Dept"])["U"].mean().rename("hist_mean_units"))

params = calib.groupby(["Store", "Dept"]).agg(
    calib_weeks=("Error_Units", "size"),
    calib_mean_pred_units=("Pred_Units", "mean"),
    err_rmse_units=("Error_Units", lambda e: float(np.sqrt(np.mean(np.square(e))))),
).reset_index().merge(hist_mean.reset_index(), on=["Store", "Dept"], how="inner")

params["Safety_Stock"] = Z * params["err_rmse_units"] * np.sqrt(LEAD_TIME_WEEKS)
params["Avg_Weekly_Demand_Units"] = params["calib_mean_pred_units"]
params["Reorder_Point"] = params["Avg_Weekly_Demand_Units"] * LEAD_TIME_WEEKS + params["Safety_Stock"]
params["EOQ"] = np.sqrt(2 * params["Avg_Weekly_Demand_Units"] * 52 * ORDER_COST / H)
params["EOQ_hist"] = np.sqrt(2 * params["hist_mean_units"] * 52 * ORDER_COST / H)
print(params.describe().round(2).T[["count", "mean", "50%", "max"]])

                          count     mean     50%       max
Store                    2924.0    22.13   22.00     45.00
Dept                     2924.0    44.10   37.00     98.00
calib_weeks              2924.0    20.71   21.00     21.00
calib_mean_pred_units    2924.0   635.40  312.86   7223.78
err_rmse_units           2924.0    84.62   47.54   1354.12
hist_mean_units          2924.0   653.25  331.56   7431.42
Safety_Stock             2924.0   197.47  110.92   3159.78
Avg_Weekly_Demand_Units  2924.0   635.40  312.86   7223.78
Reorder_Point            2924.0  1468.27  760.80  15731.12
EOQ                      2924.0   841.79  721.52   3467.04
EOQ_hist                 2924.0   857.60  742.78   3516.52


## 4. Inventory simulation (June-Oct 2012)
Policies, all with the same lead time and the same starting stock; unmet demand is lost sales.
- **A. Naive:** order the historical average weekly demand every week.
- **B. Static (s,Q):** reorder point = historical demand x lead time, no safety stock, EOQ from history.
- **C. Forecast-driven (s,Q):** reorder point = forecast demand over lead time + residual-based safety stock.

In [20]:
L = LEAD_TIME_WEEKS
assert L >= 1, "LEAD_TIME_WEEKS must be >= 1"


def run_policy(d, rule, init_stock):
    """Weekly simulation.
    Order placed at end of week t arrives at start of week t+L.
    """
    T = len(d)

    on_hand = float(init_stock)
    arrivals = np.zeros(T + L + 1)

    n_orders = 0
    hold_unit_weeks = 0.0
    lost = 0.0

    ip = on_hand

    for t in range(T):

        # Receive orders arriving this week
        on_hand += arrivals[t]

        # Satisfy demand
        sold = min(on_hand, d[t])

        # Unmet demand = lost sales
        lost += d[t] - sold

        on_hand -= sold

        # Holding inventory after satisfying demand
        hold_unit_weeks += on_hand

        # Inventory position = on-hand + outstanding orders
        ip = on_hand + arrivals[t + 1:].sum()

        # Decide whether to place a new order
        q = rule(t, ip)

        if q > 0:
            arrivals[t + L] += q
            n_orders += 1
            ip += q

    return {
        "orders": n_orders,
        "ordering_cost": n_orders * ORDER_COST,
        "holding_cost": hold_unit_weeks * H / 52.0,
        "stockout_cost": lost * STOCKOUT_COST,
        "lost_units": lost,
        "end_on_hand": on_hand,
        "end_inventory_position": ip,
    }


def reorder_rule(rop_fn, Q):
    """Order enough to raise inventory position to the reorder point."""
    
    def rule(t, ip):
        rop = rop_fn(t)

        if ip < rop:
            return Q * np.ceil((rop - ip) / Q)

        return 0.0

    return rule


# Simulation dates
sim_dates = np.sort(sim["Date"].unique())

# Store + Department parameter lookup
param_idx = params.set_index(["Store", "Dept"])

results = []
skipped = 0


for (s, dpt), g in sim.groupby(["Store", "Dept"]):

    # Skip series without parameters or with incomplete weeks
    if (s, dpt) not in param_idx.index or len(g) != len(sim_dates):
        skipped += 1
        continue

    p = param_idx.loc[(s, dpt)]

    # Skip unsuitable series
    if (
        p["hist_mean_units"] <= 0
        or p["Avg_Weekly_Demand_Units"] <= 0
        or p["calib_weeks"] < 8
    ):
        skipped += 1
        continue

    # Sort chronologically
    g = g.sort_values("Date")

    # Actual weekly demand
    d = g["Actual_Units"].to_numpy()

    # Model forecast for each week
    f = g["Pred_Units"].to_numpy()


    # ============================================================
    # FIXED FORECAST-DRIVEN LEAD-TIME DEMAND
    # ============================================================
    # At week t, calculate demand expected during the next L weeks.
    #
    # We only use forecasts for future weeks:
    # t+1, t+2, ..., t+L
    #
    # We do NOT use the actual future demand.
    # ============================================================

    f_next = np.zeros(len(f))

    for t in range(len(f)):

        start = t + 1
        end = min(t + 1 + L, len(f))

        future_forecast = f[start:end]

        if len(future_forecast) > 0:
            f_next[t] = future_forecast.sum()
        else:
            # At the very end of the simulation, use the
            # current forecast rather than an unavailable
            # future value.
            f_next[t] = f[t]


    # ============================================================
    # INITIAL INVENTORY
    # ============================================================

    hm = p["hist_mean_units"]

    init_stock = hm * (L + 1)


    # ============================================================
    # POLICY A: HISTORICAL AVERAGE ORDER
    # ============================================================

    rule_A = lambda t, ip, hm=hm: hm


    # ============================================================
    # POLICY B: STATIC REORDER POINT
    # ============================================================

    rule_B = reorder_rule(
        lambda t, hm=hm: hm * L,
        p["EOQ_hist"]
    )


    # ============================================================
    # POLICY C: FORECAST-DRIVEN REORDER POINT
    # ============================================================

    rule_C = reorder_rule(
        lambda t,
               f_next=f_next,
               ss=p["Safety_Stock"]: f_next[t] + ss,
        p["EOQ"]
    )


    # ============================================================
    # STORE-DEPARTMENT RESULTS
    # ============================================================

    row = {
        "Store": s,
        "Dept": dpt,
        "total_demand_units": d.sum()
    }


    # Run all three inventory policies
    for tag, rule in [
        ("A_naive", rule_A),
        ("B_static", rule_B),
        ("C_forecast", rule_C)
    ]:

        r = run_policy(
            d,
            rule,
            init_stock
        )

        # Store all simulation outputs
        for k, v in r.items():
            row[f"{tag}_{k}"] = v

        # Total inventory cost
        row[f"{tag}_total_cost"] = (
            r["ordering_cost"]
            + r["holding_cost"]
            + r["stockout_cost"]
        )


    # Final forecast-based reorder point
    row["C_final_reorder_point"] = (
        f_next[-1]
        + p["Safety_Stock"]
    )

    results.append(row)


# ================================================================
# FINAL SIMULATION RESULTS
# ================================================================

sim_res = pd.DataFrame(results)

print(
    f"Simulated {len(sim_res)} store-dept series "
    f"({skipped} skipped: incomplete weeks or zero demand)"
)

Simulated 2755 store-dept series (227 skipped: incomplete weeks or zero demand)


## 5. Policy comparison

In [21]:
summary = []
for tag, label in [("A_naive", "A. Naive weekly order"), ("B_static", "B. Static (s,Q), no safety stock"),
                   ("C_forecast", "C. Forecast-driven (s,Q) + safety stock")]:
    tot_dem = sim_res["total_demand_units"].sum()
    summary.append({
        "Policy": label,
        "Ordering_cost": sim_res[f"{tag}_ordering_cost"].sum(),
        "Holding_cost": sim_res[f"{tag}_holding_cost"].sum(),
        "Stockout_cost": sim_res[f"{tag}_stockout_cost"].sum(),
        "Total_cost": sim_res[f"{tag}_total_cost"].sum(),
        "Fill_rate_%": (1 - sim_res[f"{tag}_lost_units"].sum() / tot_dem) * 100,
        "Orders": sim_res[f"{tag}_orders"].sum(),
    })
summary = pd.DataFrame(summary)
print(summary.round(1).to_string(index=False))

base_A = summary.loc[0, "Total_cost"]
base_B = summary.loc[1, "Total_cost"]
cost_C = summary.loc[2, "Total_cost"]
print(f"\nC vs A (naive weekly order): {(cost_C - base_A) / base_A * 100:+.1f}% total cost")
print(f"C vs B (static s,Q):         {(cost_C - base_B) / base_B * 100:+.1f}% total cost")
print(f"Series where C is cheaper than B: {(sim_res['C_forecast_total_cost'] < sim_res['B_static_total_cost']).mean() * 100:.1f}%")

                                 Policy  Ordering_cost  Holding_cost  Stockout_cost  Total_cost  Fill_rate_%  Orders
                  A. Naive weekly order      6061000.0     7078419.8     13719183.2  26858603.0         96.7   60610
       B. Static (s,Q), no safety stock      2944000.0     3948022.5      9280858.5  16172880.9         97.7   29440
C. Forecast-driven (s,Q) + safety stock      2915600.0     5018088.2      1079557.5   9013245.7         99.7   29156

C vs A (naive weekly order): -66.4% total cost
C vs B (static s,Q):         -44.3% total cost
Series where C is cheaper than B: 59.8%


## 6. Export for Power BI

In [22]:
sim_res["Reorder_Alert"] = np.where(
    sim_res["C_forecast_end_inventory_position"] < sim_res["C_final_reorder_point"],
    "Reorder Needed", "Stock OK")

fva = test_data[["Store", "Dept", "Date", "IsHoliday", "Sales_Pos", "Pred_Sales", "Naive_LY",
                 "Actual_Units", "Pred_Units"]].rename(columns={"Sales_Pos": "Actual_Sales", "Pred_Sales": "Predicted_Sales"})
fva.to_csv(os.path.join(OUT_DIR, "forecast_vs_actual.csv"), index=False)
params.to_csv(os.path.join(OUT_DIR, "inventory_parameters.csv"), index=False)
sim_res.to_csv(os.path.join(OUT_DIR, "simulation_by_store_dept.csv"), index=False)
summary.to_csv(os.path.join(OUT_DIR, "policy_cost_summary.csv"), index=False)
comparison.to_csv(os.path.join(OUT_DIR, "model_comparison.csv"))
print("Saved to", os.path.abspath(OUT_DIR))

print("""
CAVEATS (say these in the interview):
- Unit price, holding rate, order cost, stockout cost and lead time are ASSUMED, not from the data.
- Inventory is SIMULATED; the dataset has no real stock levels.
- Cost savings are relative to the baseline policies above and depend on these assumptions.
""")

Saved to d:\Desktop\Projects\SupplyChainProject\outputs

CAVEATS (say these in the interview):
- Unit price, holding rate, order cost, stockout cost and lead time are ASSUMED, not from the data.
- Inventory is SIMULATED; the dataset has no real stock levels.
- Cost savings are relative to the baseline policies above and depend on these assumptions.

